<div align="center">

# Day 02: Model-Agnostic Explanation Methods

**Explainable Artificial Intelligence (VTR UGE 21), hands-on notebook**

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

[Day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-02) · [Lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html) · [Interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lab.html) · [PDF notes](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/Day02_Lecture_Notes.pdf)

</div>

## How to use this notebook

This notebook is the hands-on part of the day. The concepts are explained on the [lecture page](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html), and each section that applies a part of the lecture starts with a link to it. Save your own copy first with File, Save a copy in Drive, then run the cells in order with Shift+Enter. Exercises check your answer as soon as you run them. Each exercise has a reference solution in a collapsed cell: Try first, then open it. Running all cells at once also runs the reference solutions. Cells titled *Observe and reflect* ask open questions about what the code shows; they have no solution cell, and a short answer in a text cell of your copy (Insert, Text cell) is enough. Cells titled *Python note* explain a piece of the Python language used nearby. Nothing needs to be installed.

**Starting in the middle.** Each numbered section can also be opened on its own, for example from a Colab box of the lecture page. Its first code cell, *Catch-up*, runs the code of the earlier sections without their printouts and figures, in a few seconds to a few minutes. When the notebook runs from the top, the catch-up cells notice it and do nothing.

## 0. Setup

The first cell prepares the environment and defines the helper `check`, which compares an answer with the expected value and prints immediate feedback.

In [ ]:
# Environment: Colab installs the three explanation libraries; everything else is preinstalled.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'shap', 'lime', 'gemex'], check=False)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 3.6), 'axes.grid': True, 'grid.alpha': 0.3})

def check(name, value, expected, tol=1e-6):
    """Compares an answer with the expected value and prints immediate feedback."""
    if value is None:
        print(f"{name}: not answered yet. Replace None with your answer and run the cell again.")
        return False
    try:
        ok = abs(float(value) - float(expected)) <= tol * max(1.0, abs(float(expected)))
    except (TypeError, ValueError):
        ok = value == expected
    print(f"{name}: " + ("correct." if ok else "not yet. Compare your steps with the lecture and try again."))
    return ok



from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
bc = load_breast_cancer()
X = pd.DataFrame(bc.data, columns=bc.feature_names); y = pd.Series(bc.target, name="benign")
FEATURES = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, stratify=y, random_state=RANDOM_STATE)
rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr, ytr)
f = lambda D: rf.predict_proba(D[FEATURES] if hasattr(D, "columns") else D)[:, 1]
CASE = 3; x0 = Xte.iloc[CASE]
print(f"black box: test AUC {roc_auc_score(yte, f(Xte)):.3f}; case {Xte.index[CASE]} p(benign) = {f(Xte.iloc[[CASE]])[0]:.3f}")

## Python step 2: NumPy arrays, randomness and functions

NumPy, the numerical library of Python, stores numbers in arrays and computes with whole arrays at once &#91;[7](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-7)&#93;. Every method of the day perturbs columns of a data matrix, that is, changes their values on purpose, so this step works on a small matrix of five real patients and ends with a permutation importance computed by hand.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "CASE" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "python-step":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### Arrays, shape and indexing

In [ ]:
import numpy as np
Xmini = np.array([[17.99, 0.1471],
                  [20.57, 0.0702],
                  [13.54, 0.0478],
                  [ 9.50, 0.0208],
                  [13.44, 0.0203]])
ymini = np.array([1, 1, 0, 0, 1])          # 1 = malignant, 0 = benign
print(Xmini.shape, Xmini.dtype)
print("second column:", Xmini[:, 1])
print("first patient:", Xmini[0])

`np.array` turns a nested list into a two-dimensional array with five rows and two columns, which `shape` reports as `(5, 2)`. Indexing uses row and column positions: `Xmini[:, 1]` takes every row of column 1, and `Xmini[0]` takes the first row. Positions start at 0. Each row is one patient of the tumour data, with the mean radius of the cell nuclei in column 0 and their mean concave points, the number of dents in the outline, in column 1. In this small example the label 1 means malignant and 0 benign.

### Element-wise arithmetic and boolean masks

In [ ]:
pred = (Xmini[:, 1] > 0.05).astype(int)
print("predictions:", pred)
print("accuracy:", (pred == ymini).mean())
large = Xmini[:, 0] > 15
print("radius above 15:", large, "-> their concave points:", Xmini[large, 1])

A comparison between an array and a number is applied to every element and returns an array of booleans. `astype(int)` converts them to 0 and 1, and the mean of a boolean array is the share of `True` values. A boolean array used as an index, a mask, selects the matching rows.

### Randomness that can be reproduced

In [ ]:
rng = np.random.default_rng(0)
shuffled = rng.permutation(Xmini[:, 1])
print(shuffled)
again = np.random.default_rng(0).permutation(Xmini[:, 1])
print("same seed, same result:", np.array_equal(shuffled, again))

A generator created with `default_rng(seed)` produces the same sequence every time it starts from the same seed. `permutation` returns a shuffled copy. Fixing seeds is what makes an explanation reproducible.

### Functions that receive a model

In [ ]:
def rule(M):
    """The threshold rule of Day 1, written for a whole matrix."""
    return (M[:, 1] > 0.05).astype(int)

def importance(model, M, y, j, seed=0):
    rng = np.random.default_rng(seed)
    base = (model(M) == y).mean()
    Mp = M.copy()
    Mp[:, j] = rng.permutation(Mp[:, j])
    return base - (model(Mp) == y).mean()

for j, name in enumerate(["radius", "concave points"]):
    print(f"{name:15s} importance {importance(rule, Xmini, ymini, j):+.2f}")

In Python a function is a value, so `importance` can receive the model `rule` as an argument and call it. `M.copy()` protects the original matrix from the shuffle. The function is the permutation importance of the lecture with a single shuffle and the accuracy as performance measure: `base`, the accuracy on the untouched matrix, minus the accuracy after column `j` has been shuffled. For the concave points the accuracy falls from 0.8 to 0.4, an importance of 0.40. The radius receives an importance of exactly zero because the rule never reads it. `enumerate` returns positions together with the elements of a list.

**Quick check.** The array `Xmini` has shape `(5, 2)`. What is the shape of `Xmini[:, 0]`?

<details><summary>Answer</summary>

`(5,)`, a one-dimensional array with one value per patient.

</details>

### Exercises for Python step 2

**Exercise 2.1.** Store the mean radius of the five patients, the mean of the first column of `Xmini`, in `mean_radius`.

In [ ]:
mean_radius = None   # your answer

check("Exercise 2.1", mean_radius, 15.008)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
mean_radius = float(Xmini[:, 0].mean())

check("Exercise 2.1", mean_radius, 15.008)

**Exercise 2.2.** Count the patients whose radius is above 13 and whose concave points are above 0.03. Combine the two masks with the operator `&` and store the count in `n_both`.

In [ ]:
n_both = None   # your answer

check("Exercise 2.2", n_both, 3)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
n_both = int(((Xmini[:, 0] > 13) & (Xmini[:, 1] > 0.03)).sum())

check("Exercise 2.2", n_both, 3)

**Exercise 2.3.** One shuffle is a random draw. Average the importance of the concave points, column 1, over the seeds 0 to 9 and store the mean in `imp_mean`.

In [ ]:
imp_mean = None   # your answer

check("Exercise 2.3", imp_mean, 0.44)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
imp_mean = float(np.mean([importance(rule, Xmini, ymini, 1, seed=s) for s in range(10)]))

check("Exercise 2.3", imp_mean, 0.44)

## 1. The black box to explain

*Lecture: [Explaining a model that cannot be read](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#explaining-a-model-that-cannot-be-read)*

A random forest of 300 trees for the tumour data. `f(D)` returns its probability of benign for every row of a data frame `D`. **Python move: a function as the interface.** The methods of the day only call `f` or the equivalent `rf.predict_proba`. The one exception is TreeSHAP in sections 4, 5, 7 and 8, which reads the trees themselves.

**From the formulas to the code.** `f(Xte)` is the model $f$ of the lecture, applied to every test tumour: the average of the 300 tree answers, read as the probability of benign. `(p_te > 0.5) == yte` marks the correct decisions, and its mean is the accuracy. The setup cell printed the second measure with `roc_auc_score`: the area under the receiver operating characteristic curve, abbreviated AUC. From here on the labels are those of the data set, 1 for benign and 0 for malignant, the reverse of the small example in the Python step. `CASE = 3` selects the patient of the day, row 518 of the data, and `x0` holds the measurements of this patient.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "imp_mean" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-1":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
p_te = f(Xte)
print(f"random forest of 300 trees: test accuracy {((p_te > 0.5) == yte).mean():.3f}")
print(f"test patient {CASE}: probability of benign {p_te[CASE]:.3f}, diagnosis {['malignant', 'benign'][yte.iloc[CASE]]}")
plt.figure(figsize=(6.5, 3))
for cls, col, name in ((0, "#c0392b", "malignant"), (1, "#2980b9", "benign")):
    plt.hist(p_te[yte.values == cls], bins=20, range=(0, 1), alpha=.6, color=col, label=name)
plt.axvline(p_te[CASE], color="k", ls="--", lw=1, label=f"patient {CASE}")
plt.xlabel("predicted probability of benign"); plt.ylabel("test tumours"); plt.legend(fontsize=8); plt.show()

**Python note.** `p_te[yte.values == cls]` keeps the probabilities of the test tumours of one class: The comparison gives an array of `True` and `False`, a mask, and indexing with the mask keeps the entries where it is `True`. `.values` turns the pandas column into a NumPy array in the same order.

**Observe and reflect.** A few test tumours receive probabilities between 0.3 and 0.7. Would you let the forest decide alone for these tumours, and what would you want to show a doctor about them?

## 2. Which features matter? Permutation importance

*Lecture: [Which features matter? Permutation importance](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#which-features-matter-permutation-importance)*

**From the formulas to the code.** In `permutation_importance_scratch`, `base` is the performance $M(f, X, y)$ on the untouched test table, here the AUC. `rng.permutation(Xp[col].values)` shuffles one column of a copy, and `base - metric(y, f(Xp))` is the drop $D_{j,r}$ of one shuffle. `n_repeats=10` is the number of shuffles $R$. `np.mean(drops)` is the importance $\mathrm{PI}_j$, and `np.std(drops)` is its standard deviation $\mathrm{SD}_j$, which divides by $R$ as in the lecture and is drawn as the orange error bar.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "p_te" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-2":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def permutation_importance_scratch(f, X, y, metric=roc_auc_score, n_repeats=10, seed=0):
    rng = np.random.default_rng(seed); base = metric(y, f(X)); out = {}
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy(); Xp[col] = rng.permutation(Xp[col].values)
            drops.append(base - metric(y, f(Xp)))
        out[col] = (np.mean(drops), np.std(drops))
    return pd.DataFrame(out, index=["mean", "std"]).T.sort_values("mean", ascending=False)

pfi = permutation_importance_scratch(f, Xte, yte)
t = pfi.head(10).iloc[::-1]
plt.barh(t.index, t["mean"], xerr=t["std"], color="#34495e", ecolor="#e67e22")
plt.xlabel("drop in test AUC when the column is shuffled"); plt.title("Permutation importance"); plt.show()
pfi.head(6).round(4)

**Python note.** The function receives the model as its first argument `f` and the measure as `metric`, with `roc_auc_score` as the default value, so the same code serves any model and any measure. `X.copy()` makes a new table, so that a shuffle never changes `X` itself. `pd.DataFrame(out, index=["mean", "std"])` makes one column per measurement, and `.T` turns the table so that every measurement gets a row.

The next cell reads the measurement with the largest permutation importance, the first row of the sorted table `pfi`.

In [ ]:
top_pfi = pfi.index[0]
print(f"largest permutation importance: {top_pfi}, {pfi['mean'].iloc[0]:.4f} with a spread of {pfi['std'].iloc[0]:.4f}")

**Observe and reflect.** Run section 2 again with `seed=1` in the call of `permutation_importance_scratch`. Does the same measurement come first, and what does the answer say about the order of the first places in the figure?

## 3. How does a feature act? Partial dependence and ICE &#91;[2](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-2)&#93;

*Lecture: [How does a feature act? Partial dependence and ICE](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#how-does-a-feature-act-partial-dependence-and-ice)*

**From the formulas to the code.** An individual conditional expectation (ICE) curve is the thin line of one tumour, and the partial dependence is the thick line. `PartialDependenceDisplay.from_estimator` of scikit-learn computes both formulas of the lecture: For each grid value $v$ it writes $v$ into the column of the measurement for every test tumour and asks the forest for the probability of benign. `kind="both"` draws the ICE curves and their average. The grid has 100 values between about the 5th and the 95th percentile of the measurement, the average runs over all 143 test tumours, and `subsample=60` draws 60 randomly chosen ICE curves. The short marks on the horizontal axis are the deciles, the nine values that cut the tumours into ten groups of equal size.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "top_pfi" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-3":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
from sklearn.inspection import PartialDependenceDisplay
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
for a, feat in zip(ax, ["worst concave points", "worst radius"]):
    PartialDependenceDisplay.from_estimator(rf, Xte, [feat], kind="both", subsample=60, random_state=RANDOM_STATE, ax=a,
                                            ice_lines_kw={"alpha": .25}, pd_line_kw={"color": "#c0392b", "lw": 3})
plt.tight_layout(); plt.show()

**Explore.** Move the controls and watch the result update. If the controls do not appear, run the cell again.

In [ ]:
import ipywidgets as W
def pdp_of(feature="worst texture"):
    fig, a = plt.subplots(figsize=(6, 3.4))
    PartialDependenceDisplay.from_estimator(rf, Xte, [feature], kind="both", subsample=60, random_state=RANDOM_STATE, ax=a,
                                            ice_lines_kw={"alpha": .25}, pd_line_kw={"color": "#c0392b", "lw": 3})
    plt.show()
W.interact(pdp_of, feature=W.Dropdown(options=FEATURES, value="worst texture"));

**Observe and reflect.** Choose a measurement whose ICE curves cross or spread out. What does such a pattern say about the forest, and why does the partial dependence alone not show it?

## 4. Why this patient? LIME and SHAP side by side &#91;[3](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-3), [4](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-4)&#93;

*Lecture: [Why this case? Shapley values and SHAP](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#why-this-case-shapley-values-and-shap)*

**From the formulas to the code.** LIME stands for local interpretable model-agnostic explanations, and SHAP for Shapley additive explanations. `explain_instance` draws 5000 samples, weights them with the kernel $\pi_x(z)$ and fits the weighted ridge regression of the lecture on six features (`num_features=6`), and `lime_w` holds the weights $w_j$, slopes per standard deviation. The `lime` library differs from the five steps of the lecture in two points: It draws the samples around the average training tumour, and it takes the square root of the kernel, with the width $h = 0.75 \sqrt{30} = 4.11$. `shap.TreeExplainer(rf)` computes the Shapley values $\phi_j$ of all 30 measurements for every test tumour, `sv[..., 1]` keeps those for the class benign, and `shap_w` is the row of the patient. `tree_shap.expected_value` holds the base value $\phi_0$ for both classes, and Exercise A takes the one for benign. The two charts are read differently: A SHAP bar is a contribution, while a LIME bar is a slope, so a red LIME bar says that the probability of benign falls when the measurement grows, not that the value of the patient speaks for malignant.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "fig" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-4":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
import shap
from lime.lime_tabular import LimeTabularExplainer
lime_exp = LimeTabularExplainer(Xtr.values, feature_names=FEATURES, class_names=["malignant", "benign"],
                                discretize_continuous=False, random_state=RANDOM_STATE)
lime_res = lime_exp.explain_instance(x0.values, lambda Z: rf.predict_proba(pd.DataFrame(Z, columns=FEATURES)), num_features=6)
lime_w = pd.Series(dict(lime_res.as_list()))
tree_shap = shap.TreeExplainer(rf)
sv = tree_shap.shap_values(Xte)
sv = sv[..., 1] if np.ndim(sv) == 3 else sv
shap_w = pd.Series(sv[CASE], index=FEATURES)
top_shap = shap_w.reindex(shap_w.abs().sort_values(ascending=False).index)[:6]
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
for a, (name, w) in zip(ax, (("LIME", lime_w), ("SHAP", top_shap))):
    w = w.iloc[::-1]; a.barh(range(len(w)), w.values, color=["#2980b9" if v > 0 else "#c0392b" for v in w])
    a.set_yticks(range(len(w))); a.set_yticklabels([s[:24] for s in w.index], fontsize=8); a.axvline(0, color="grey", lw=.8)
    a.set_title(f"{name} for patient {CASE}: towards benign (blue), malignant (red)", fontsize=9)
plt.tight_layout(); plt.show()
shared = set(lime_w.index) & set(top_shap.index)
print(f"features in both top-six lists: {len(shared)} of 6 -> {sorted(shared)}")

**Python note.** `lambda Z: rf.predict_proba(pd.DataFrame(Z, columns=FEATURES))` wraps the forest in a function of a plain array, the form in which LIME passes its samples. `set(lime_w.index) & set(top_shap.index)` intersects two sets: `&` keeps the names that occur in both.

**Exercise A.** Check the additivity of SHAP: The SHAP values of the patient plus the expected value of the explainer give the prediction. Store `float(shap_w.sum() + base)` in `rebuilt`, where `base` is `np.ravel(tree_shap.expected_value)[-1]`.

In [ ]:
rebuilt = None   # your answer

check("Exercise A", rebuilt, float(f(Xte.iloc[[CASE]])[0]), tol=1e-3)

In [ ]:
#@title Reference solution (run this cell; open it only after trying)
base = np.ravel(tree_shap.expected_value)[-1]
rebuilt = float(shap_w.sum() + base)

check("Exercise A", rebuilt, float(f(Xte.iloc[[CASE]])[0]), tol=1e-3)

## 5. From local to global: the SHAP summary

*Lecture: [Why this case? Shapley values and SHAP](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#why-this-case-shapley-values-and-shap)*

**From the formulas to the code.** `np.abs(sv).mean(0)` is the mean absolute SHAP value $G_j$ of the lecture: the absolute values of one column of `sv`, averaged over the 143 test tumours. `shap.summary_plot` draws every single value $\phi_j^{(i)}$ as a dot. Its horizontal position is the SHAP value, its colour is the value of the measurement, red for high and blue for low, and the rows are sorted by $G_j$.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "rebuilt" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-5":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
shap.summary_plot(sv, Xte, max_display=10, show=False, plot_size=(8, 4.2))
plt.title("SHAP values of every test patient: one dot per patient and measurement", fontsize=9); plt.show()
global_shap = pd.Series(np.abs(sv).mean(0), index=FEATURES).sort_values(ascending=False)
print("mean absolute SHAP value, top five:"); print(global_shap.head(5).round(4))

**Observe and reflect.** Find a measurement whose red dots lie on both sides of zero. What could make a high value push some tumours towards benign and others towards malignant?

## 6. What would change the decision? A counterfactual &#91;[5](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-5)&#93;

*Lecture: [What would change the decision?](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#what-would-change-the-decision)*

**From the formulas to the code.** `target` is the other class, here 0 for malignant, and `goal` holds the medians $m_j$ of the training tumours of that class. `z[j] + (goal[j] - z[j]) / 4` is the step $z_j + (m_j - z_j) / 4$ of the lecture. In every round, `min(features, key=...)` tries this step for each of the eight candidates, the measurements with the largest mean absolute SHAP value, and keeps the one that lowers the probability of benign most. The loop stops when the decision has flipped, after 25 rounds at the latest. This greedy search does not minimise the distance of the counterfactual formula, so it returns a counterfactual that need not be the nearest one.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "global_shap" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-6":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
def counterfactual(x, features, steps=25):
    """Greedy search: change one of the given features at a time, towards the median of the other class, until the decision flips."""
    target = 1 - int(f(pd.DataFrame([x]))[0] > 0.5)
    goal = Xtr[ytr == target].median(); z = x.copy(); changed = []
    for _ in range(steps):
        if int(f(pd.DataFrame([z]))[0] > 0.5) == target:
            break
        best = min(features, key=lambda j: f(pd.DataFrame([z.where(z.index != j, z[j] + (goal[j] - z[j]) / 4)]))[0] * (1 if target == 0 else -1))
        z[best] = z[best] + (goal[best] - z[best]) / 4
        if best not in changed: changed.append(best)
    return z, changed

candidates = list(global_shap.index[:8])
cf, changed = counterfactual(x0, candidates)
table = pd.DataFrame({"patient": x0[changed], "counterfactual": cf[changed]})
table["change"] = table.counterfactual - table.patient
print(f"probability of benign: patient {f(pd.DataFrame([x0]))[0]:.3f}, counterfactual {f(pd.DataFrame([cf]))[0]:.3f}")
print(table.round(3).to_string())

**Python note.** `min(features, key=...)` returns the feature for which the function after `key=` is smallest, here the probability after one step. `z.where(z.index != j, new)` returns a copy of `z` in which only the entry `j` is replaced by `new`. `break` leaves the loop as soon as the decision has flipped, and `if best not in changed` records each changed feature only once.

The next cell counts the measurements that the counterfactual changed.

In [ ]:
n_changed = len(changed)
print(f"the counterfactual changed {n_changed} of {len(FEATURES)} measurements: {changed}")

**Observe and reflect.** The counterfactual is valid and sparse, but its worst perimeter does not fit its worst radius. Which check of plausibility would you add to the function `counterfactual` before its result is shown to a patient?

## 7. Your field: the application switch

*Lecture: [Explanations in your field](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#explanations-in-your-field)*

Change `FIELD` to `chemistry`, `manufacturing`, `energy` or `civil` and run the cell again.

**From the formulas to the code.** `load_field` returns the cases, the outcomes and the name of the positive class of one field, and a new random forest of 300 trees is trained on 70 percent of the cases. `permutation_importance` of scikit-learn is the formula of section 2 with ten shuffles. Its performance measure is the accuracy, where section 2 used the AUC. `sv_f` holds the SHAP values of the test cases for the positive class: `np.abs(sv_f).mean(0)` is the mean absolute SHAP value of every feature, and `sv_f[i]` explains test case `i`.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "n_changed" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-7":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

In [ ]:
FIELD = "medicine"          # choose: medicine, chemistry, manufacturing, energy or civil

def load_field(field, n=1500, seed=RANDOM_STATE):
    """A yes or no problem from one field: features, outcome and the name of the positive class."""
    rng = np.random.default_rng(seed)
    if field == "medicine":
        d = load_breast_cancer(as_frame=True); return d.data, d.target, "benign tumour"
    if field == "chemistry":
        from sklearn.datasets import load_wine
        d = load_wine(as_frame=True); return d.data, (d.target == 0).astype(int), "wine of cultivar 1"
    if field == "manufacturing":
        F = pd.DataFrame({"current_A": rng.normal(180, 20, n), "voltage_V": rng.normal(24, 2, n),
                          "speed_mm_s": rng.normal(6, 1.2, n), "gap_mm": rng.gamma(2, .4, n), "plate_temp_C": rng.normal(40, 15, n)})
        z = 0.04 * (F.current_A - 180) - 0.9 * (F.speed_mm_s - 6) - 1.8 * (F.gap_mm - .8) + 0.02 * (F.plate_temp_C - 40)
        return F, pd.Series((z + rng.normal(0, 1, n) > 0).astype(int)), "sound weld"
    if field == "energy":
        F = pd.DataFrame({g: rng.lognormal(m, .6, n) for g, m in (("H2", 3.5), ("CH4", 3.0), ("C2H2", 0.5), ("C2H4", 2.5), ("CO", 5.0))})
        z = 0.8 * np.log(F.C2H2) + 0.6 * np.log(F.H2 / 30) + 0.5 * np.log(F.C2H4 / 12) - 0.2 * np.log(F.CO / 150)
        return F, pd.Series((z + rng.normal(0, .6, n) > 0.6).astype(int)), "fault in the transformer"
    if field == "civil":
        F = pd.DataFrame({"cement": rng.uniform(150, 500, n), "water": rng.uniform(140, 230, n), "slag": rng.uniform(0, 200, n),
                          "superplasticizer": rng.uniform(0, 15, n), "age_days": rng.choice([7, 14, 28, 56, 90], n)})
        strength = (0.09 * F.cement - 0.15 * F.water + 0.04 * F.slag + 0.7 * F.superplasticizer
                    + 9 * np.log(F.age_days / 7) + rng.normal(0, 4, n))
        return F, pd.Series((strength > 30).astype(int)), "mix reaching 30 MPa"
    raise ValueError(f"unknown field: {field}")

from sklearn.datasets import load_breast_cancer
from sklearn.inspection import permutation_importance
Xf, yf, positive = load_field(FIELD)
Xa, Xb, ya, yb = train_test_split(Xf, yf, test_size=0.3, stratify=yf, random_state=RANDOM_STATE)
model_f = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1).fit(Xa, ya)
print(f"{FIELD}: positive class '{positive}', test accuracy {model_f.score(Xb, yb):.3f}")
pi = permutation_importance(model_f, Xb, yb, n_repeats=10, random_state=RANDOM_STATE)
imp = pd.Series(pi.importances_mean, index=Xf.columns).sort_values()
sv_f = shap.TreeExplainer(model_f).shap_values(Xb)
sv_f = sv_f[..., 1] if np.ndim(sv_f) == 3 else sv_f
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
imp.tail(8).plot.barh(ax=ax[0], color="#2980b9"); ax[0].set_title("permutation importance", fontsize=9)
plt.sca(ax[1]); shap.summary_plot(sv_f, Xb, max_display=8, show=False, plot_size=None)
ax[1].set_title("SHAP values of the test cases", fontsize=9); plt.tight_layout(); plt.show()

**Observe and reflect.** In the simulated fields the true rule is known. Does the feature that the rule leaves out come last in both panels, and how would you check a ranking on real data, where no rule is known?

## 8. Going further (optional)

*Lecture: [Going further (optional)](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#going-further-optional)*

The limits of the methods of the day, in six experiments.

**From the formulas to the code.** Each of the six experiments below starts with its own note on how the code follows the formulas of the lecture.

**Started here?** If you opened the notebook at this section, run the next cell once. It runs the code of the earlier sections for you, without their printouts and figures, which takes from a few seconds to a few minutes, and it does nothing when they have already run. The Colab menu Runtime, Run before, which runs every cell above the selected one, works as well.

In [ ]:
#@title Catch-up: run this cell once if you start the notebook here
if "sv_f" in globals():
    print("The earlier sections have already run. Continue with the next cell.")
else:
    import json, urllib.request
    import matplotlib.pyplot as plt
    from IPython import get_ipython
    from IPython.utils.capture import capture_output
    _cu_cells = json.load(urllib.request.urlopen("https://raw.githubusercontent.com/utkukose/xai-veltech-NB-lecture/main/days/day-02/NB02_model_agnostic_methods.ipynb"))["cells"]
    _cu_ip, _cu_err = get_ipython(), None
    with capture_output():
        for _cu_c in _cu_cells:
            if _cu_c.get("id") == "sec-8":
                break
            if _cu_c["cell_type"] == "code" and _cu_c.get("metadata", {}).get("catchup", True):
                try:
                    exec(_cu_ip.transform_cell("".join(_cu_c["source"])), _cu_ip.user_ns)
                except Exception as _cu_e:
                    _cu_err = _cu_e
                    break
    plt.close("all")
    print("The code of the earlier sections has run. Continue with the next cell." if _cu_err is None else
          f"The catch-up stopped with an error ({_cu_err}). Run the notebook from the top instead.")

### A duplicated feature

The mean radius and the mean perimeter of a tumour are near copies of each other. `X['mean radius'].corr(X['mean perimeter'])` is their correlation coefficient $r$ from the lecture. `fit_and_pfi` trains a forest on four measurements, or on three without the twin, and applies `permutation_importance_scratch` of section 2 to it. `twin_ratio` divides the importance of the mean radius without its twin by its importance next to the twin.

In [ ]:
WITH = ["mean radius", "mean perimeter", "worst texture", "mean smoothness"]
WITHOUT = ["mean radius", "worst texture", "mean smoothness"]
def fit_and_pfi(cols):
    m = RandomForestClassifier(n_estimators=300, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1).fit(Xtr[cols], ytr)
    g = lambda D: m.predict_proba(D[cols])[:, 1]
    return roc_auc_score(yte, g(Xte[cols])), permutation_importance_scratch(g, Xte[cols], yte)
auc_w, pfi_w = fit_and_pfi(WITH); auc_o, pfi_o = fit_and_pfi(WITHOUT)
print(f"correlation of the twins: {X['mean radius'].corr(X['mean perimeter']):.3f}")
print(f"with both twins: AUC {auc_w:.4f}, importance of mean radius {pfi_w.loc['mean radius', 'mean']:.4f}")
print(f"twin removed:    AUC {auc_o:.4f}, importance of mean radius {pfi_o.loc['mean radius', 'mean']:.4f}")
twin_ratio = pfi_o.loc["mean radius", "mean"] / max(pfi_w.loc["mean radius", "mean"], 1e-6)
print(f"the importance changed by a factor of {twin_ratio:.1f} while the AUC barely moved")

### LIME from scratch, the library version and its fidelity

`lime_scratch` follows the five steps of the lecture. `rng.normal(x.values, sd, ...)` draws the samples $z_k$ around the patient, and `Zs` holds their distances from the patient in standard deviations, feature by feature. `w` is the kernel weight $\pi_x(z_k)$ with the width `kw`, which is $h = 0.75 \sqrt{d}$. `Lars`, short for least angle regression, a method that adds features to a linear model one at a time, selects the six features. `Ridge(alpha=1.0)` is the weighted fit with the penalty factor $\alpha = 1$, and `r2` is the local fidelity $R^2$. The second cell runs the `lime` library twice, with and without its default step that replaces every measurement by one of four classes (`discretize_continuous`), and prints the $R^2$ that the library reports as `score`.

In [ ]:
from sklearn.linear_model import Ridge, Lars
def lime_scratch(f, x, X_ref, n_samples=1000, kernel_width=None, n_features=6, seed=0):
    rng = np.random.default_rng(seed); sd = X_ref.std().values
    Z = rng.normal(x.values, sd, size=(n_samples, len(x)))
    fz = f(pd.DataFrame(Z, columns=X_ref.columns)); Zs = (Z - x.values) / sd
    kw = kernel_width or 0.75 * np.sqrt(len(x)); w = np.exp(-(np.linalg.norm(Zs, axis=1) ** 2) / kw ** 2)
    sel = np.argsort(np.abs(Lars(n_nonzero_coefs=n_features).fit(Zs * w[:, None], fz * w).coef_))[::-1][:n_features]
    g = Ridge(alpha=1.0).fit(Zs[:, sel], fz, sample_weight=w); pred = g.predict(Zs[:, sel])
    r2 = 1 - np.sum(w * (fz - pred) ** 2) / np.sum(w * (fz - np.average(fz, weights=w)) ** 2)
    return pd.Series(g.coef_, index=X_ref.columns[sel]).sort_values(key=abs, ascending=False), r2

lime_w, lime_r2 = lime_scratch(f, x0, Xtr)
print(f"LIME from scratch: local fidelity R^2 = {lime_r2:.3f}"); print(lime_w.round(4).to_string())

In [ ]:
from lime.lime_tabular import LimeTabularExplainer
r2_lib = {}
for disc in (True, False):
    ex = LimeTabularExplainer(Xtr.values, feature_names=FEATURES, class_names=["malignant", "benign"],
                              discretize_continuous=disc, random_state=RANDOM_STATE)
    e = ex.explain_instance(x0.values, lambda A: rf.predict_proba(A), num_features=6, num_samples=1000)
    r2_lib[disc] = e.score
    print(f"lime library, discretize_continuous={disc}: local R^2 = {e.score:.3f}")

### Twenty runs of LIME

`runs` repeats `lime_scratch` with the seeds 0 to 19. The table `M` has one row per run and one column for every measurement that was selected at least once, with 0 where a run did not select it. `freq` counts the runs in which a measurement appears, and the box plot shows the weights of the eight most frequent ones.

In [ ]:
runs = [lime_scratch(f, x0, Xtr, seed=s)[0] for s in range(20)]
M = pd.DataFrame(runs).fillna(0.0); freq = (M != 0).sum().sort_values(ascending=False)
top = freq.index[:8]
plt.figure(figsize=(8, 3.6)); plt.boxplot([M[c].values for c in top], tick_labels=[c[:18] for c in top])
plt.axhline(0, color="grey", lw=.6); plt.xticks(rotation=35, ha="right", fontsize=7)
plt.ylabel("LIME weight"); plt.title("Twenty LIME runs, one patient, one model"); plt.tight_layout(); plt.show()
unstable = freq[(freq > 0) & (freq < 20)]
print(f"features that appear in some runs and not in others: {len(unstable)}")

### Exact Shapley values, KernelSHAP and TreeSHAP

`v_of_S(S, x)` is the value function $v(S)$ of the lecture: a copy of the baseline row `bg`, the medians of the training tumours, in which the features of the coalition `S` take the values of the case. In `exact_shapley`, `w` is the weight $s! \, (d - s - 1)! / d!$ of a coalition with $s$ features among $d = 4$ players, and `total` adds the weighted changes $v(S \cup \{j\}) - v(S)$ up to $\phi_j$. `payoffs` holds the gain $v(F) - v(\varnothing)$ of every test tumour, and `CASE_SH` is the tumour with the largest one in absolute value. KernelSHAP is asked for 200 coalitions (`nsamples=200`), more than the 14 that exist between the empty and the full one, so it computes them all and its values are exact here. The second cell runs TreeSHAP on all 30 measurements and compares the mean absolute SHAP values with the permutation importance of section 2 by the rank correlation of Spearman, `rho`.

In [ ]:
from itertools import combinations
from math import factorial
import shap
SUB = ["worst radius", "worst concave points", "worst texture", "mean smoothness"]
bg = Xtr.median()
def v_of_S(S, x):
    row = bg.copy()
    for j in S: row[j] = x[j]
    return float(f(pd.DataFrame([row])[FEATURES])[0])
def exact_shapley(x, players):
    n, phi = len(players), {}
    for j in players:
        rest = [p for p in players if p != j]; total = 0.0
        for k in range(n):
            for S in combinations(rest, k):
                w = factorial(len(S)) * factorial(n - len(S) - 1) / factorial(n)
                total += w * (v_of_S(list(S) + [j], x) - v_of_S(list(S), x))
        phi[j] = total
    return pd.Series(phi)
v0 = v_of_S([], Xte.iloc[0])
payoffs = np.array([v_of_S(SUB, Xte.iloc[k]) - v0 for k in range(len(Xte))])
CASE_SH = int(np.argmax(np.abs(payoffs))); x_sh = Xte.iloc[CASE_SH]
phi = exact_shapley(x_sh, SUB)
print(f"payoff for the main case: {payoffs[CASE]:+.3f} (nothing to divide); using case {Xte.index[CASE_SH]}, payoff {payoffs[CASE_SH]:+.3f}")
print("exact Shapley values:"); print(phi.round(4).to_string())
print(f"efficiency: sum {phi.sum():+.4f} against v(N) - v(empty) {v_of_S(SUB, x_sh) - v0:+.4f}")
f_sub = lambda A: rf.predict_proba(pd.DataFrame(np.repeat(bg.values[None, :], len(A), 0), columns=FEATURES)
                                   .assign(**{c: A[:, k] for k, c in enumerate(SUB)})[FEATURES])[:, 1]
ks = shap.KernelExplainer(f_sub, np.array([bg[SUB].values]))
kernel = np.ravel(ks.shap_values(x_sh[SUB].values.reshape(1, -1), nsamples=200, silent=True)[0])
cmp_shap = pd.DataFrame({"exact": phi[SUB].values, "KernelSHAP": kernel}, index=SUB)
cmp_shap["abs error"] = (cmp_shap.exact - cmp_shap.KernelSHAP).abs(); print(cmp_shap.round(5))

In [ ]:
tex = shap.TreeExplainer(rf)
SV = tex.shap_values(Xte); SV = SV[..., 1] if SV.ndim == 3 else SV
glob = pd.Series(np.abs(SV).mean(0), index=FEATURES).sort_values(ascending=False)
rho = glob.corr(pfi["mean"].reindex(glob.index), method="spearman")
print(f"TreeSHAP for the whole test set; Spearman between mean |SHAP| and permutation importance: {rho:.2f}")
glob.head(8).round(4)

### A greedy anchor &#91;[6](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lecture.html#ref-6)&#93;

`band(c)` returns the quarter of the training values of measurement `c` that contains the value of the patient, one condition of a rule. `evaluate(conds)` returns the precision and the coverage of a rule. The coverage is `m.mean()`, the share of training tumours inside the rule. For the precision, 600 training tumours are drawn at random, their columns named in the rule are overwritten with values of tumours inside the rule, and the share with the decision of the patient is counted. The loop adds the condition with the highest precision until `precision_target=0.95` is reached or the rule has `max_conditions=4` conditions.

In [ ]:
def anchor_search(f, x, X_ref, precision_target=0.95, max_conditions=4, n_samples=600, n_bins=4, seed=0, candidates=None):
    rng = np.random.default_rng(seed); pred0 = int(f(pd.DataFrame([x]))[0] > .5)
    cols = list(candidates) if candidates is not None else list(X_ref.columns)
    edges = {c: np.quantile(X_ref[c], np.linspace(0, 1, n_bins + 1)) for c in cols}
    def band(c):
        e = edges[c]; k = int(np.clip(np.searchsorted(e, x[c], side="right") - 1, 0, n_bins - 1))
        return float(e[k]), float(e[k + 1])
    def evaluate(conds):
        m = np.ones(len(X_ref), bool)
        for c, (lo, hi) in conds.items(): m &= (X_ref[c].values >= lo) & (X_ref[c].values <= hi)
        if m.sum() < 5: return 0.0, float(m.mean())
        Z = X_ref.iloc[rng.integers(0, len(X_ref), n_samples)].copy().reset_index(drop=True); pool = X_ref[m]
        for c in conds: Z[c] = pool[c].values[rng.integers(0, len(pool), n_samples)]
        return float(np.mean((f(Z) > .5).astype(int) == pred0)), float(m.mean())
    conds, rule = {}, []; prec, cov = evaluate(conds)
    for _ in range(max_conditions):
        best = None
        for c in cols:
            if c in conds: continue
            trial = dict(conds); trial[c] = band(c); p_, cv_ = evaluate(trial)
            if best is None or p_ > best[1]: best = ((c, trial[c]), p_, cv_)
        (c, b), prec, cov = best; conds[c] = b; rule.append(f"{c} in [{b[0]:.2f}, {b[1]:.2f}]")
        if prec >= precision_target: break
    return rule, prec, cov, pred0
rule, prec, cov, pred0 = anchor_search(f, x0, Xtr, candidates=pfi["mean"].head(10).index)
print("prediction:", "benign" if pred0 else "malignant")
for r in rule: print("   IF", r)
print(f"precision {prec:.3f}, coverage {cov:.3f}")

### Slope or contribution?

`conv = w * (Xte.iloc[k] - med) / sd` is the conversion $c_j = w_j (x_j - m_j) / s_j$ of the lecture, with the medians `med` and the standard deviations `sd` of the training tumours. The loop visits every fourth test tumour, fits `lime_scratch` on all 30 measurements and compares signs on the ten measurements with the largest absolute SHAP values. `sign_raw` is the share of equal signs for the raw weights, and `sign_conv` for the converted ones.

In [ ]:
med, sd = Xtr.median(), Xtr.std(); raw_hits = conv_hits = tot = 0
for k in range(0, len(Xte), 4):
    w, _ = lime_scratch(f, Xte.iloc[k], Xtr, n_features=30); w = w.reindex(FEATURES).fillna(0.0)
    sv = pd.Series(SV[k], index=FEATURES); top10 = sv.abs().sort_values(ascending=False).index[:10]
    conv = w * (Xte.iloc[k] - med) / sd
    raw_hits += int((np.sign(sv[top10]) == np.sign(w[top10])).sum())
    conv_hits += int((np.sign(sv[top10]) == np.sign(conv[top10])).sum()); tot += len(top10)
sign_raw, sign_conv = raw_hits / tot, conv_hits / tot
print(f"over {tot // 10} cases and their top-10 features:")
print(f"   sign agreement, raw LIME slope against SHAP: {sign_raw:.0%}")
print(f"   sign agreement, slope converted to a contribution: {sign_conv:.0%}")

## Application challenges

Each challenge transfers the ideas of the day to a field of study. Choose the one closest to your programme and solve it in a copy of this notebook.

| Field | Challenge |
|---|---|
| Computer Science and Engineering | Implement the conditional variant of permutation importance, which shuffles a feature only within bins of its strongest correlate, and compare it with the permutation importance of section 2 for the twin features of section 8. |
| Electronics and Communication Engineering | Generate a synthetic signal-classification dataset with two redundant spectral features and show the correlation trap of section 8, the duplicated feature, on it. |
| Electrical and Electronics Engineering | For a load-forecasting regression model of your choice, plot the partial dependence (PDP) and the accumulated local effects (ALE) of temperature and explain the difference to an operator in three sentences. |
| Mechanical Engineering | Use the counterfactual search of section 6 to find the smallest change of operating conditions that moves a simulated machine from the failure class to the safe class, and check its plausibility. |
| Biomedical Engineering | Repeat the twenty-seed LIME experiment with 5000 samples per run and report whether the set of top-six features stabilises. |
| Civil Engineering | Compute exact Shapley values for a four-feature model of concrete strength that you train on synthetic mix data, and verify the efficiency property. |
| Aeronautical Engineering | Design an anchor for a simple rule-based flight-envelope classifier and report its precision and coverage. |
| Biotechnology | For a model with a known interaction, such as two genes acting only together, compare Shapley values with LIME slopes and explain the difference. |

## After the notebook

The self-assessment and the reflection are in the [interactive lab](https://utkukose.github.io/xai-veltech-NB-lecture/days/day-02/lab.html), tabs *Check yourself* and *Reflect and export*. The daily task and the research assignment are on the [day overview](https://github.com/utkukose/xai-veltech-NB-lecture/tree/main/days/day-02).

---

<div align="center">

**Prof. Dr. Utku Kose**

Full Professor, Department of Computer Engineering, Süleyman Demirel University, Isparta, Türkiye  
Founding Director, AI Application and Research Center (YAZEM), Süleyman Demirel University  
Head of the Computer Science Division, Department of Computer Engineering, Süleyman Demirel University  
Additional affiliations: University of North Dakota (USA), Universidad Panamericana (Mexico City, Mexico), Vel Tech University (Chennai, India)  
IEEE Senior Member, ACM Professional Member

[ORCID 0000-0002-9652-6415](https://orcid.org/0000-0002-9652-6415) | [utkukose.com](https://www.utkukose.com) | [github.com/utkukose](https://github.com/utkukose)

[utkukose@sdu.edu.tr](mailto:utkukose@sdu.edu.tr) | [utku.kose@und.edu](mailto:utku.kose@und.edu) | [ukose@up.edu.mx](mailto:ukose@up.edu.mx) | [utkukose@gmail.com](mailto:utkukose@gmail.com)

</div>